# Detección de ejercicios y técnica — Práctica 3

Reconoce **sentadilla, lagartija y curl de bíceps** y dice si cada repetición está **bien** o qué **error** tiene.

**Cómo funciona**
1. MediaPipe detecta 33 puntos del cuerpo en cada cuadro.
2. Con esos puntos se calculan 6 ángulos: rodilla, cadera, codo, hombro, inclinación del torso y línea hombro-cadera-tobillo.
3. **Modelo de ejercicio:** clasifica ventanas de ~1 s de ángulos (media, desviación, mínimo y máximo).
4. Las repeticiones se cortan con el ángulo principal (rodilla en sentadilla, codo en lagartija y curl).
5. **Modelo de técnica** (uno por ejercicio): clasifica cada repetición como `bien` o el tipo de error.

Se comparan Random Forest, SVM y MLP con validación cruzada agrupada por persona (o por video) y se guarda el mejor.

**Estructura en Google Drive**
```
MyDrive/deteccion_ejercicios/
  videos/
    horta_andrea/
      sentadilla__bien__1.mp4
      sentadilla__poco_profunda__1.mp4
      lagartija__cadera_caida__1.mp4
      curl__balanceo__1.mp4
      ...
    otro_integrante/ ...
  pruebas/        <- videos para probar el modelo ya entrenado
```
Etiquetas válidas: **sentadilla**: `bien`, `poco_profunda`, `espalda_inclinada` · **lagartija**: `bien`, `cadera_caida`, `poco_profunda` · **curl**: `bien`, `balanceo`, `rango_incompleto`.

Si la carpeta te la compartieron, en Drive haz clic derecho → *Organizar → Agregar acceso directo* a "Mi unidad" para que Colab la vea.

## 1. Preparar el entorno

In [ ]:
REPO = "https://github.com/AndreaHortaHernandez/Deteccion-de-ejercicios.git"
import os
if os.path.exists("/content/Deteccion-de-ejercicios"):
    !git -C /content/Deteccion-de-ejercicios pull -q
else:
    !git clone -q $REPO /content/Deteccion-de-ejercicios
%cd /content/Deteccion-de-ejercicios
# --no-deps: evita que MediaPipe instale una segunda copia de OpenCV encima de la de Colab
!pip install -q --no-deps mediapipe==1.2.2
!pip install -q flatbuffers
import pose
pose.asegurar_modelo_pose()
print("Listo")

## 2. Conectar Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/deteccion_ejercicios"   # cámbialo si su carpeta tiene otro nombre
VIDEOS = f"{BASE}/videos"
DATOS = f"{BASE}/datos"        # aquí se guardan los puntos de pose (CSV); así no se recalculan
MODELOS = f"{BASE}/modelos"
SALIDAS = f"{BASE}/salidas"
for p in (DATOS, MODELOS, SALIDAS):
    os.makedirs(p, exist_ok=True)
!ls "$VIDEOS"

## 3. Extraer la pose de los videos
Solo procesa videos nuevos o modificados. Al final lista los videos con problemas (nombre mal escrito, persona fuera de cuadro…).

In [ ]:
!python extraer_videos.py --videos "$VIDEOS" --datos "$DATOS"

## 4. Entrenar y evaluar
Muestra, para cada modelo, la comparación de algoritmos, el reporte por clase y la matriz de confusión.

In [ ]:
!python entrenar.py --datos "$DATOS" --modelos "$MODELOS" --salidas "$SALIDAS"

In [ ]:
import glob
from IPython.display import Image, display
for png in sorted(glob.glob(f"{SALIDAS}/confusion_*.png")):
    display(Image(png))

## 5. Probar con un video
Usa un video que **no** se haya usado para entrenar (ponlo en `pruebas/` o súbelo aquí).

In [ ]:
VIDEO_PRUEBA = f"{BASE}/pruebas/prueba1.mp4"   # o deja "" para subir uno desde tu compu

if not VIDEO_PRUEBA:
    from google.colab import files
    subido = files.upload()
    VIDEO_PRUEBA = "/content/" + next(iter(subido))

!python analizar_video.py --video "$VIDEO_PRUEBA" --salida /content/analizado.mp4 --modelos "$MODELOS"
# convertir a H.264 para que el navegador lo pueda reproducir
!ffmpeg -y -loglevel error -i /content/analizado.mp4 -vcodec libx264 -pix_fmt yuv420p /content/analizado_h264.mp4
from IPython.display import Video
Video("/content/analizado_h264.mp4", embed=True, width=640)

## 6. Cámara en vivo
Colab pide permiso para usar la cámara. Colócate **de perfil** y con el cuerpo completo en cuadro.
Va a unos cuantos cuadros por segundo (cada cuadro viaja al servidor de Colab), así que haz las repeticiones a ritmo normal o un poco lento.
Haz clic sobre el video para detener.

In [ ]:
import importlib, colab_camara
importlib.reload(colab_camara)
analizador = colab_camara.correr(MODELOS)